In [1]:
from __future__ import annotations
 
import json
import os
import time
from dataclasses import dataclass, field
 
import httpx
from openai import OpenAI
 
MODEL = os.environ.get("MODEL", "gpt-4o-mini")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [2]:
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

load_dotenv()
import httpx._models as m

_orig = m._normalize_header_value
def _patched(value, encoding=None):
    try:
        return _orig(value, encoding)
    except UnicodeEncodeError:
        print("BAD HEADER VALUE:", repr(value))
        raise
m._normalize_header_value = _patched

def get_weather(city: str) -> dict:
    """도시의 현재 날씨를 조회한다.

    Args:
        city: 도시 이름. 예 '서울'
    """
    return {"city": city, "temp_c": 27, "condition": "맑음"}

agent = LlmAgent(
    name="check",
    model="gemini-3.6-flash",
    instruction="날씨를 물으면 반드시 get_weather 툴을 써라.",
    tools=[get_weather],
)

async def main():
    runner = InMemoryRunner(agent=agent, app_name="check")
    s = await runner.session_service.create_session(app_name="check", user_id="u1")
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text="서울 날씨 어때?")]),
    ):
        for p in (ev.content.parts if ev.content else []) or []:
            if p.function_call:
                print("TOOL  ", p.function_call.name, dict(p.function_call.args))
            if p.function_response:
                print("RESULT", p.function_response.response)
            if p.text:
                print("TEXT  ", p.text.strip())

await main()

c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\tools\function_tool.py:96: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


TOOL   get_weather {'city': '서울'}
RESULT {'city': '서울', 'temp_c': 27, 'condition': '맑음'}
TEXT   현재 서울의 날씨는 **맑음**이며, 기온은 **27℃**입니다.


In [3]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

agent = LlmAgent(
    name="searcher",
    model="gemini-3.6-flash",
    instruction="웹 검색으로 사실을 확인하고, 이름,위치,전화번호,내용을 딱 정리해라 근거 링크를 함께 제시해라.",
    tools=[google_search],
)

async def main():
    runner = InMemoryRunner(agent=agent, app_name="s")
    s = await runner.session_service.create_session(app_name="s", user_id="u1")
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text="26.07.25 기준 마산 창원 근교 행사 및 축제")]),
    ):
        if ev.content:
            for p in ev.content.parts:
                if p.text:
                    print(p.text.strip())
        gm = getattr(ev, "grounding_metadata", None)
        if gm:
            print("\n[검색어]", gm.web_search_queries)
            for c in (gm.grounding_chunks or []):
                print("-", c.web.title, "|", c.web.uri)

await main()

2026년 7월 25일 기준, **마산·창원 지역 및 근교에서 진행되는 주요 축제 및 행사 정보**입니다.

---

### 1. [마산] 경남 마산로봇랜드 여름 축제 '2026 쿨썸머 로캉스'
* **이름**: 2026 쿨썸머 로캉스
* **위치**: 경상남도 창원시 마산합포구 구산면 로봇랜드로 250 (마산로봇랜드)
* **전화번호**: 055-214-6000
* **내용**: 
  * 물총 싸움 '워터워 대전', '슈팅 워터카', '워터락 파티' 등 다양한 참여형 워터 프로그램 운영
  * **7월 25일 특별 일정**: 전문 DJ가 참여하는 EDM 공연(하루 2회) 개최
  * 야외 물놀이장 '워터플레이 그라운드' 상시 운영 (월요일 휴장)
* **근거 링크**: [이트레블뉴스 - 경상남도, 로봇랜드에서 여름 축제 2026 쿨썸머 로캉스 개최](https://www.e-travelnews.com/news/article.html?no=147721)

---

### 2. [창원 근교 - 거제] 2026 거제여름해양축제
* **이름**: 2026 거제여름해양축제
* **위치**: 경상남도 거제시 일운면 지세포해안로 43 (지세포항 수변공원 및 거제해양레포츠센터 일원)
* **전화번호**: 055-639-8100 (거제해양관광개발공사)
* **내용**: 
  * 7월 25일~26일 양일간 진행되는 대표 해양 여름 축제
  * 대형 워터풀, 워터슬라이드, 밤 시간대 물대포 연출 '오션밤' 프로그램 운영
  * **7월 25일 개막식 공연**: 박서진, 코요태, 미란이, tripleS 출연
  * 요트 투어, 패들보드(SUP), 카약 등 해양레포츠 체험 제공
* **근거 링크**: [뉴스프리존 - '2026 거제여름해양축제' 7월 25~26일 개최](https://www.newsfreezone.co.kr/news/articleView.html?idxno=589132)

---

### 3. [창원 근교 - 밀양] 2026 밀양공연예술축제
* **이름**: 202

In [4]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

agent = LlmAgent(
    name="basic",
    model="gemini-3.6-flash",
    instruction="너는 친절한 도우미다.",
)
user_prompt = "한국의 수도는?"
runner = InMemoryRunner(agent=agent, app_name="a")
session = runner.session_service.create_session_sync(app_name="a", user_id="u1")
out = runner.run(user_id="u1",session_id=session.id,new_message=types.Content(role="user", parts=[types.Part(text=user_prompt)]))

Deprecated. Please migrate to the async method.


In [5]:
next(out).content.parts

[Part(
   text='한국의 수도는 **서울**입니다. 😊',
   thought_signature=b'\x12\x8c\x05\n\x89\x05\x01\x11M2\x0fg\x97\xfd-\x14\xcc\x1a~\xf4\x1b\xc5\xecN\xa9\xafV\x95\xc55\xfd\x10\xfds~\xb7/\\ \xa7o\x87\x83\xf7)\xf8\x9b\x8e\xc6!\x83$\xa4F\xe8\xa8)p\r1\xb5c\x8a\xbap\xb0\x13\x01\xda\xf4\x04l\x1eY\x17K\xcc\xc1Za\xbfnS\xf0h\xda\x15N\xbb4\x1a\x15\xa7\xd6\xc7\x86\xe7...'
 )]

In [6]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

agent = LlmAgent(
    name="basic",
    model="gemini-3.6-flash",
    instruction="너는 친절한 도우미다.",
)

runner = InMemoryRunner(agent=agent, app_name="a")

def ask(user_prompt: str) -> str:
    session = runner.session_service.create_session_sync(app_name="a", user_id="u1")
    out = ""
    for ev in runner.run(
        user_id="u1",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=user_prompt)]),
    ):
        if ev.is_final_response() and ev.content:
            out = "".join(p.text or "" for p in ev.content.parts)
    return out

print(ask("한국의 수도는?"))

Deprecated. Please migrate to the async method.


한국의 수도는 **서울**입니다. 😊


In [7]:
from pydantic import BaseModel, Field
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

# ── 받고 싶은 JSON 형태를 클래스로 정의 (이게 곧 스키마) ──
class PlaceQuery(BaseModel):
    queries: list[str] = Field(description="검색에 쓸 쿼리 3~5개")
    region: str = Field(description="사용자가 말한 지역")
    category: str = Field(description="음식점/카페/관광 중 하나")

SYSTEM = "너는 사용자 요청을 검색 쿼리로 바꾸는 도우미다. 지역명을 반드시 쿼리에 포함해라."

agent = LlmAgent(
    name="planner",
    model="gemini-3.6-flash",
    instruction=SYSTEM,
    output_schema=PlaceQuery,          # ← JSON 강제. tools와 동시 사용 불가
)

async def ask(user_prompt: str) -> PlaceQuery:
    runner = InMemoryRunner(agent=agent, app_name="p")
    s = await runner.session_service.create_session(app_name="p", user_id="u1")
    result = None
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=user_prompt)]),
    ):
        if ev.is_final_response() and ev.content:
            text = "".join(p.text or "" for p in ev.content.parts)
            result = PlaceQuery.model_validate_json(text)   # 문자열 → 검증된 객체
    return result

out = await ask("청주 성안길 근처 국밥집 찾아줘")
print(type(out))
print(out.queries)
print(out.region, "/", out.category)

<class '__main__.PlaceQuery'>
['청주 성안길 국밥 맛집', '청주 성안길 돼지국밥 순대국밥', '청주 성안길 국밥 추천', '청주 성안길 근처 국밥집']
청주 성안길 / 음식점


In [10]:
from pydantic import BaseModel, Field
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
MODEL = "gemini-3.6-flash"

# ══════════════════════════════════════════════════
# 스키마 (각 단계의 출력 모양)
# ══════════════════════════════════════════════════
class Queries(BaseModel):
    queries: list[str] = Field(description="검색 쿼리 3~5개. 지역명 포함")

class Card(BaseModel):
    name: str = Field(description="장소/행사 이름")
    reason: str = Field(description="추천 근거 한 줄")
    source: str = Field(description="근거 URL")

class Cards(BaseModel):
    cards: list[Card] = Field(description="추천 카드 목록")

# ══════════════════════════════════════════════════
# 에이전트 3개
# ══════════════════════════════════════════════════
planner = LlmAgent(
    name="planner", model=MODEL,
    instruction="사용자 요청을 검색 쿼리로 바꿔라. 지역명을 반드시 포함해라.",
    output_schema=Queries,
)

searcher = LlmAgent(          # tools는 output_schema와 공존 불가 → 얘만 검색
    name="searcher", model=MODEL,
    instruction="주어진 쿼리들로 웹을 검색하고, 찾은 장소/행사를 근거 URL과 함께 정리해라.",
    tools=[google_search],
)

extractor = LlmAgent(
    name="extractor", model=MODEL,
    instruction=(
        "검색 결과 텍스트에서 추천 카드를 뽑아라. "
        "근거 URL이 없는 항목은 제외해라. 이름/근거/출처를 채워라."
    ),
    output_schema=Cards,
)

# ══════════════════════════════════════════════════
# 각 에이전트를 부르는 헬퍼 (동기)
# ══════════════════════════════════════════════════
def run_agent(agent, text: str) -> str:
    runner = InMemoryRunner(agent=agent, app_name="wh")
    s = runner.session_service.create_session_sync(app_name="wh", user_id="u1")
    out = ""
    for ev in runner.run(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        if ev.is_final_response() and ev.content:
            out = "".join(p.text or "" for p in ev.content.parts)
    return out

# ══════════════════════════════════════════════════
# 흐름 실행 + goal/loop
# ══════════════════════════════════════════════════
def pipeline(user_request: str, goal: int = 6, max_round: int = 2):
    all_cards, seen = [], set()

    for rnd in range(1, max_round + 1):
        print(f"\n{'='*50}\n[라운드 {rnd}]")

        # [1] 쿼리 생성
        q_json = run_agent(planner, user_request)
        queries = Queries.model_validate_json(q_json).queries
        print("쿼리:", queries)

        # [2] 검색 (쿼리 합쳐서 한 번에)
        search_result = run_agent(searcher, "다음 쿼리들로 검색해줘:\n" + "\n".join(queries))
        print("검색결과 길이:", len(search_result), "자")

        # [3] 추출
        c_json = run_agent(extractor, search_result)
        cards = Cards.model_validate_json(c_json).cards

        # 중복 제거하며 누적
        before = len(all_cards)
        for c in cards:
            if c.name not in seen:
                seen.add(c.name)
                all_cards.append(c)
        gained = len(all_cards) - before
        print(f"신규 카드: {gained}개 (누적 {len(all_cards)})")

        # [4] 골/루프 판정
        if len(all_cards) >= goal:
            print(f"→ 종료: 목표 달성 ({len(all_cards)}/{goal})"); break
        if gained == 0:
            print("→ 종료: 무진전"); break
        if rnd == max_round:
            print("→ 종료: 예산 소진")
        else:
            user_request += " (더 다양하게 찾아줘)"   # 축 변경(간단 버전)

    return all_cards

# ── 실행 ──
cards = pipeline("청주 성안길 근처 국밥집 추천해줘")
print(f"\n{'='*50}\n최종 카드 {len(cards)}개:")
for c in cards:
    print(f"- {c.name} | {c.reason} | {c.source}")

Deprecated. Please migrate to the async method.



[라운드 1]


Deprecated. Please migrate to the async method.


쿼리: ['청주 성안길 국밥 맛집', '청주 성안길 근처 순대국밥 돼지국밥', '청주 성안길 유명 국밥집 추천', '청주 성안길 가성비 국밥 추천']


Deprecated. Please migrate to the async method.


...

신규 카드: 5개 (누적 5)

[라운드 2]


Deprecated. Please migrate to the async method.


쿼리: ['청주 성안길 국밥 맛집', '청주 성안길 돼지국밥 순대국밥 추천', '청주 성안길 근처 24시 국밥집', '청주 성안길 소머리국밥 콩나물국밥 맛집', '청주 성안길 로컬 현지인 국밥 추천']


Deprecated. Please migrate to the async method.


검색결과 길이: 1234 자
신규 카드: 0개 (누적 5)
→ 종료: 무진전

최종 카드 5개:
- 새가덕순대 | 잡내 없이 진하고 깔끔한 국물의 40년 전통 순대국밥 노포 | https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFurFjcmsmUCHfKO4dPirrWPGiaPXn56C0xFlrtaYwzK5s3RfWqSOIsKzggl3VOz48NgISA5Umu-iXHebej_1DCMi8gTE_QnTNPMcNoJLsyTeX6UjYILDDrvO5pee2bbcXsK4Kuim-QYRNAk_BNQ5wHDb6c3T0Q91nXREEpEg66cjs42P0bkIy7hNK1jAI_mkpc2o1rYJPRcV2_qt0bII1UOjq0hA8Nls3giYFjtBYdyrIchC_wwIctGvPNyMMsfGBgZ4e2Y5tMqESFXQSwauTqh8OAy8SOLMHIxjqBUR-xP8wJedh8rHC6KMlQWE_hEr4pqviIfDlaQKHq_VBTNkA_T6fj6efnopXAbvDgZ4qENsYx4l_VASMJZSCN3hBGDofzsqtXqPpxUh4V2RDUAJhNYj9dZmm533ytAk9UY2M-L6NqHwwNzdjUPKaGsQPZaH3wLDFy1RzYUIpS_a5x_e9wFc0
- 남주동해장국 | 소고기와 선지, 양을 푸짐하게 넣은 깊고 매콤한 백년가게 전통 선지해장국 | https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbc34_iDDRCf9Le1bryKEcuUVMVwhI_DMdPYenurVzC2LE1UdqxuwAw_nDttYZnjWg8-n_q2XKmComCatntUQmyBPIj9mqRY8ONrL9QZYqCCwG_lD4
- 육수당 청주북문로점 | 순대국밥, 수육국밥 등 메뉴가 다양하고 성안길 내 혼밥과 가성비 식사에 적합한 국밥집 | https://vertexaisearch.cloud.google.com/grounding-api-r

In [15]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
MODEL = "gemini-3.6-flash"

# 서브 3개 — google_search만 가짐 (다른 툴/함수 섞지 않음)
restaurant_agent = LlmAgent(
    name="restaurant_agent", model=MODEL,
    description="음식점·맛집 검색 담당",
    instruction="음식점을 검색해서 이름과 근거 URL을 정리해라.",
    tools=[google_search],
)
cafe_agent = LlmAgent(
    name="cafe_agent", model=MODEL,
    description="카페·디저트 검색 담당",
    instruction="카페를 검색해서 이름과 근거 URL을 정리해라.",
    tools=[google_search],
)
festival_agent = LlmAgent(
    name="festival_agent", model=MODEL,
    description="지역 축제·행사·공연 검색 담당",
    instruction="지역 행사를 검색해서 이름·기간·근거 URL을 정리해라.",
    tools=[google_search],
)

# 코디네이터 — 툴 없음. sub_agents로 순수 위임만.
coordinator = LlmAgent(
    name="coordinator", model=MODEL,
    instruction=(
        "사용자 요청을 분석해 가장 적합한 서브 에이전트에게 위임(transfer)해라. "
        "맛집이면 restaurant_agent, 카페면 cafe_agent, 행사면 festival_agent. "
        "직접 답하지 말고 반드시 위임해라."
    ),
    sub_agents=[restaurant_agent, cafe_agent, festival_agent],
)

def run_with_log(user_request: str):
    runner = InMemoryRunner(agent=coordinator, app_name="wh")
    s = runner.session_service.create_session_sync(app_name="wh", user_id="u1")
    print(f"[USER] {user_request}\n{'='*55}")
    final = ""
    for ev in runner.run(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=user_request)]),
    ):
        author = ev.author
        for p in (ev.content.parts if ev.content else []) or []:
            if getattr(p, "function_call", None):
                fc = p.function_call
                if fc.name == "transfer_to_agent":
                    print(f"  ↪ [위임] {author} → {fc.args.get('agent_name')}")
                else:
                    print(f"  🔍 [툴] {author}: {fc.name}")
            elif getattr(p, "function_response", None):
                print(f"  ← [결과] {p.function_response.name}")
            elif p.text and p.text.strip():
                print(f"  [{author}] {p.text.strip()[:120]}")
        if ev.is_final_response() and ev.content:
            final = "".join(x.text or "" for x in ev.content.parts)
    print(f"\n{'='*55}\n[최종]\n{final}")
    return final

run_with_log("어머님 모시고 갈만한 곳 창원에 추천해줄래?")

Deprecated. Please migrate to the async method.


[USER] 어머님 모시고 갈만한 곳 창원에 추천해줄래?


Tools at indices [0] are not compatible with automatic function calling (AFC). AFC is disabled. If AFC is intended, please include python callables in the tool list, and do not include function declaration and MCP server in the tool list.


  ↪ [위임] coordinator → restaurant_agent
  ← [결과] transfer_to_agent


Node execution failed with exception
Traceback (most recent call last):
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\workflow\_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\workflow\_base_node.py", line 166, in run
    async for item in agen:
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\agents\llm_agent.py", line 601, in _run_impl
    async for event in agen:
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\workflow\_llm_agent_wrapper.py", line 370, in run_llm_agent_as_node
    async for event in run_iter:
  File "c:\VSCode\wh_poc\.venv\Lib\site-packages


[최종]



''

In [17]:
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
MODEL = "gemini-3.6-flash"

# ══════════════════════════════════════════════════
# 자율 검색 에이전트 — 루프를 스스로 돈다
# ══════════════════════════════════════════════════
search_agent = LlmAgent(
    name="autonomous_searcher",
    model=MODEL,
    instruction="""너는 지역 행사/장소를 찾아주는 검색 에이전트다.

사용자 요청에서 다음을 파악해라:
- 관심사 (축제/카페/맛집 등)
- 동반자 (부모님이면 접근성·주차·좌석을 신경써라)
- 지역 (근교라고 하면 인접 시군까지 넓혀도 된다)
- 기간 (있으면 그 기간에 열리는 것만)

검색 전략:
- 먼저 핵심 지역으로 검색한다.
- 결과가 부족하거나 조건(기간/동반자)에 안 맞으면,
  근교로 범위를 넓히거나 다른 각도로 다시 검색해라.
- 목표는 조건에 맞는 후보 6개 이상이다.
- 6개를 채우거나 더 찾기 어렵다고 판단되면 멈춰라.

최종 답변:
- 각 후보를 [이름 / 기간 / 부모님 적합성 / 근거 URL] 형식으로.
- 근거 URL 없는 건 넣지 마라. 못 찾으면 솔직히 말해라.""",
    tools=[google_search],
)

# ══════════════════════════════════════════════════
# 실행 + 로그 (에이전트가 몇 번 검색하는지 관찰)
# ══════════════════════════════════════════════════
def run_autonomous(user_request: str):
    runner = InMemoryRunner(agent=search_agent, app_name="wh")
    s = runner.session_service.create_session_sync(app_name="wh", user_id="u1")

    print(f"[USER] {user_request}\n{'='*60}")
    search_count = 0
    final = ""

    for ev in runner.run(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=user_request)]),
    ):
        for p in (ev.content.parts if ev.content else []) or []:
            if getattr(p, "function_call", None):
                search_count += 1
                fc = p.function_call
                # google_search의 쿼리가 args에 담김
                print(f"  🔍 [검색 {search_count}] {dict(fc.args)}")
            elif getattr(p, "function_response", None):
                print(f"  ← [검색결과 수신]")
            elif p.text and p.text.strip():
                # 중간 사고 과정도 텍스트로 나올 수 있음
                snippet = p.text.strip()
                if not ev.is_final_response():
                    print(f"  💭 {snippet[:100]}")
        if ev.is_final_response() and ev.content:
            final = "".join(x.text or "" for x in ev.content.parts)

    print(f"\n{'='*60}")
    print(f"[에이전트가 스스로 검색한 횟수: {search_count}회]")
    print(f"\n[최종 답변]\n{final}")
    return final

# ── 실행 ──
run_autonomous("부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이.")

Deprecated. Please migrate to the async method.


[USER] 부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이.

[에이전트가 스스로 검색한 횟수: 0회]

[최종 답변]
창원 및 창원 근교에서 **25일~27일 기간** 동안 부모님을 모시고 가기 좋은 축제 및 전시/행사 추천 목록입니다. 부모님의 동선, 주차 편의성, 관람 좌석 및 접근성을 고려해 구성했습니다.

---

1. **[진주 강주연꽃 문화축제 / 2026.07.25 ~ 2026.07.26 / 부모님 적합성 / https://www.jjinews.asia/default/index_view_page.phtml?idx=202758]**
   - **설명:** 강주연못 수변을 따라 평탄한 산책 데크와 쉼터가 조성되어 있어 경사 없이 편안하게 걸으며 연꽃을 감상할 수 있습니다. 무대공연과 전국가요제 좌석이 마련되어 있어 앉아서 관람하시기 좋습니다.

2. **[밀양공연예술축제 / 2026.07.21 ~ 2026.08.01 / 부모님 적합성 / https://www.mycf.or.kr/art/festival.php]**
   - **설명:** 밀양아리나 및 밀양아리랑아트센터 실내·외 공연장의 지정 좌석에서 시원하게 연극, 뮤지컬, 국악 등의 무대를 관람하실 수 있어 체력적 부담이 매우 적습니다.

3. **[거제여름해양축제 / 2026.07.25 ~ 2026.07.26 / 부모님 적합성 / http://www.gndomin.com/news/articleView.html?idxno=396425]**
   - **설명:** 지세포항 수변공원에서 바다 야경과 바람을 즐기실 수 있으며, 어르신들에게 인기가 높은 가수(박서진 등)의 트로트 및 무대 공연을 좌석에 앉아 감상할 수 있습니다.

4. **[사천 삼천포대교공원 토요상설무대 프러포즈 / 2026.07.25 ~ 2026.08.15 / 부모님 적합성 / http://www.jjinews.asia/default/index_view_page.phtml?idx=202877]**


'창원 및 창원 근교에서 **25일~27일 기간** 동안 부모님을 모시고 가기 좋은 축제 및 전시/행사 추천 목록입니다. 부모님의 동선, 주차 편의성, 관람 좌석 및 접근성을 고려해 구성했습니다.\n\n---\n\n1. **[진주 강주연꽃 문화축제 / 2026.07.25 ~ 2026.07.26 / 부모님 적합성 / https://www.jjinews.asia/default/index_view_page.phtml?idx=202758]**\n   - **설명:** 강주연못 수변을 따라 평탄한 산책 데크와 쉼터가 조성되어 있어 경사 없이 편안하게 걸으며 연꽃을 감상할 수 있습니다. 무대공연과 전국가요제 좌석이 마련되어 있어 앉아서 관람하시기 좋습니다.\n\n2. **[밀양공연예술축제 / 2026.07.21 ~ 2026.08.01 / 부모님 적합성 / https://www.mycf.or.kr/art/festival.php]**\n   - **설명:** 밀양아리나 및 밀양아리랑아트센터 실내·외 공연장의 지정 좌석에서 시원하게 연극, 뮤지컬, 국악 등의 무대를 관람하실 수 있어 체력적 부담이 매우 적습니다.\n\n3. **[거제여름해양축제 / 2026.07.25 ~ 2026.07.26 / 부모님 적합성 / http://www.gndomin.com/news/articleView.html?idxno=396425]**\n   - **설명:** 지세포항 수변공원에서 바다 야경과 바람을 즐기실 수 있으며, 어르신들에게 인기가 높은 가수(박서진 등)의 트로트 및 무대 공연을 좌석에 앉아 감상할 수 있습니다.\n\n4. **[사천 삼천포대교공원 토요상설무대 프러포즈 / 2026.07.25 ~ 2026.08.15 / 부모님 적합성 / http://www.jjinews.asia/default/index_view_page.phtml?idx=202877]**\n   - **설명:** 삼천포대교공원 내 해상무대 관람석에 편안히 앉아 남해 바다 야경과 함께 트로트·감성 음악회를 관람할 수 있습

In [1]:
import asyncio, nest_asyncio
from pydantic import BaseModel, Field
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()
MODEL = "gemini-3.6-flash"

# ══════════════════════════════════════════════════
# 1) 검색 에이전트 4개 — 각자 도메인, google_search만 (output_schema 없음)
# ══════════════════════════════════════════════════
def make_searcher(name, domain_desc):
    return LlmAgent(
        name=name, model=MODEL,
        instruction=(
            f"너는 {domain_desc} 검색 전문이다.\n"
            "반드시 google_search를 사용해 실제로 검색해라. 기억으로 답하지 마라.\n"
            "사용자 요청의 지역·기간·동반자 조건에 맞는 곳을 찾아라.\n"
            "찾은 각 항목을 [이름 / 기간(있으면) / 특징 / 근거URL] 로 정리해라.\n"
            "검색 결과에 없는 건 지어내지 마라."
        ),
        tools=[google_search],
    )

searchers = {
    "restaurant": make_searcher("restaurant_searcher", "식당·맛집"),
    "cafe":       make_searcher("cafe_searcher", "카페·디저트"),
    "festival":   make_searcher("festival_searcher", "축제·행사·공연"),
    "spot":       make_searcher("spot_searcher", "관광지·미술관·요즘 뜨는 명소"),
}

# ══════════════════════════════════════════════════
# 2) 메인 에이전트 — output_schema로 최종 카드 (툴 없음)
# ══════════════════════════════════════════════════
class Card(BaseModel):
    domain: str = Field(description="restaurant/cafe/festival/spot 중 하나")
    name: str
    period: str = Field(description="행사 기간. 없으면 '상시' 또는 ''")
    reason: str = Field(description="부모님 동반 등 조건에 맞는 이유 한 줄")
    source: str = Field(description="근거 URL")

class Feed(BaseModel):
    cards: list[Card] = Field(description="도메인별 추천 카드 모음")

main_agent = LlmAgent(
    name="main", model=MODEL,
    instruction=(
        "4개 도메인의 검색 결과를 받아 추천 카드로 정리해라.\n"
        "- 근거 URL이 없는 항목은 제외해라.\n"
        "- 사용자 조건(기간/동반자/지역)에 안 맞는 건 빼라.\n"
        "- 각 도메인에서 고르게 뽑되, 조건에 맞는 것 위주로."
    ),
    output_schema=Feed,
)

# ══════════════════════════════════════════════════
# 3) 실행 헬퍼
# ══════════════════════════════════════════════════
async def run_one(agent, text):
    runner = InMemoryRunner(agent=agent, app_name="wh")
    s = await runner.session_service.create_session(app_name="wh", user_id="u1")
    out, n_search = "", 0
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        for p in (ev.content.parts if ev.content else []) or []:
            if getattr(p, "function_call", None):
                n_search += 1
        if ev.is_final_response() and ev.content:
            out = "".join(x.text or "" for x in ev.content.parts)
    return out, n_search

async def run_feed(user_request: str):
    print(f"[USER] {user_request}\n{'='*60}")

    # ── 4개 검색 병렬 실행 ──
    print("[1] 4개 도메인 병렬 검색 시작...")
    results = await asyncio.gather(*[
        run_one(ag, user_request) for ag in searchers.values()
    ])
    for (dom, (text, n)) in zip(searchers.keys(), results):
        print(f"  ✓ {dom:12} 검색 {n}회, 결과 {len(text)}자")

    # ── 검색 결과 합쳐서 메인에 전달 ──
    combined = "\n\n".join(
        f"### {dom} 검색결과 ###\n{text}"
        for dom, (text, _) in zip(searchers.keys(), results)
    )
    print("\n[2] 메인 에이전트가 카드로 정리 중...")
    feed_json, _ = await run_one(main_agent, f"사용자 요청: {user_request}\n\n{combined}")
    feed = Feed.model_validate_json(feed_json)

    # ── 출력 ──
    print(f"\n{'='*60}\n[최종 카드 {len(feed.cards)}개]")
    for c in feed.cards:
        print(f"  [{c.domain}] {c.name} | {c.period} | {c.reason}")
        print(f"       {c.source}")
    return feed

# ── 실행 ──
feed = await run_feed(
    "부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이."
)

[USER] 부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이.
[1] 4개 도메인 병렬 검색 시작...
  ✓ restaurant   검색 0회, 결과 1433자
  ✓ cafe         검색 0회, 결과 1419자
  ✓ festival     검색 0회, 결과 1502자
  ✓ spot         검색 0회, 결과 1355자

[2] 메인 에이전트가 카드로 정리 중...

[최종 카드 4개]
  [restaurant] 사천시 삼천포항 자연산 전어축제 | 2026.07.24 ~ 2026.07.28 | 창원 근교에서 싱싱한 제철 전어 요리를 맛보고 바다 풍경을 둘러볼 수 있어 부모님 식도락 여행으로 제격입니다.
       https://www.gyeongnam.go.kr/tourfestival/index.gyeong?menuCd=DOM_000000902001000000
  [cafe] 2026 토요상설무대 프러포즈 | 2026.07.25 ~ 2026.08.15 | 7월 25일 개막 공연에서 남해 바다 야경과 함께 부모님 취향의 감성 음악 무대를 관람하기 좋습니다.
       http://www.gnnews.org/news/articleView.html?idxno=218320
  [festival] 제26회 밀양공연예술축제 | 2026.07.21 ~ 2026.08.01 | 창원 근교 밀양에서 열리며 부모님과 함께 야외 산책을 겸해 다양한 연극과 공연 예술을 여유롭게 감상할 수 있습니다.
       https://www.gyeongnam.go.kr/tour/index.gyeong?menuCd=DOM_000000902002000000
  [spot] 김해가야테마파크 멀티미디어 불꽃쇼 | 2026.07.25 ~ 2026.08.15 | 7월 25일 야간 불꽃쇼와 아름다운 빛의 산책로가 조성되어 있어 부모님을 모시고 시원한 밤 산책과 야경을 즐기기 좋습니다.
       https://gaya.gopf.or.kr/


In [2]:
async def run_one_debug(agent, text):
    runner = InMemoryRunner(agent=agent, app_name="wh")
    s = await runner.session_service.create_session(app_name="wh", user_id="u1")
    out = ""
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        gm = getattr(ev, "grounding_metadata", None)
        if gm:
            print("  검색어:", gm.web_search_queries)
            for c in (gm.grounding_chunks or []):
                print("   -", c.web.title, "|", c.web.uri[:60])
        if ev.is_final_response() and ev.content:
            out = "".join(x.text or "" for x in ev.content.parts)
    return out

# festival 하나만 디버그
res = await run_one_debug(searchers["festival"], 
    "부모님 모시고 갈 축제, 창원 근교, 25~27일")

  검색어: ['경남 축제 25일 26일 27일', '창원 근교 축제 25일 27일']
   - gyeongnam.go.kr | https://vertexaisearch.cloud.google.com/grounding-api-redire
   - jinju.news | https://vertexaisearch.cloud.google.com/grounding-api-redire
   - visitkorea.or.kr | https://vertexaisearch.cloud.google.com/grounding-api-redire
   - changwon.go.kr | https://vertexaisearch.cloud.google.com/grounding-api-redire


In [3]:
import os, httpx, html, re
from dotenv import load_dotenv

load_dotenv()
CID = os.environ["NAVER_CLIENT_ID"]
CSEC = os.environ["NAVER_CLIENT_SECRET"]

def _strip(t):
    return html.unescape(re.sub(r"<[^>]+>", "", t or "")).strip()

def naver_search(query: str, kind: str = "blog", n: int = 5) -> list[dict]:
    """네이버 검색. kind: blog / local / webkr / news"""
    r = httpx.get(
        f"https://openapi.naver.com/v1/search/{kind}.json",
        headers={"X-Naver-Client-Id": CID, "X-Naver-Client-Secret": CSEC},
        params={"query": query, "display": n, "sort": "sim"},
        timeout=5.0,
    )
    r.raise_for_status()
    items = r.json().get("items", [])
    return [
        {
            "title": _strip(it.get("title", "")),
            "desc": _strip(it.get("description", ""))[:150],
            "url": it.get("link", ""),
        }
        for it in items
    ]

# 테스트
for it in naver_search("어머니 모시고 가기 좋은 카페 창원", kind="blog", n=5):
    print("•", it["title"])
    print("  ", it["desc"])
    print("  ", it["url"])
    print()

• [창원 여행 추천] 바다가... 부모님 모시고 가기 좋은 곳  + 카페...
   ('ࠏ' ) 이상으로 포스팅을 마칩니다♥ https://m.blog.naver.com/230708gw/224199999406 갔다가 들릴 카페 찾으시는... 구산면여행 #창원부모님여행 #부모님모시고가기좋은곳 #국내힐링여행 #경남여행 #경남가볼만한곳...
   https://blog.naver.com/230708gw/224201120441

• 창원카페 부모님 모시고 가기 좋은 오션뷰 카페플라야
   까지는 아니어도 잔잔한 파도소리 들으며 힐링 하기 좋은 거 같아요 창원 부모님 모시고 가기 좋은 카페플라야 메뉴예요 플라야 시그니처 메뉴는 없지만 커피, 디카페인, 티, 쥬스등 다양한 음료가 있어요 또...
   https://blog.naver.com/s002s003/223774714511

• 창원 식물 카페 앵지밭골그집 부모님 모시고 가기 좋은 정원...
   창원 식물 카페 앵지밭골그집 글,사진,영상 / ⓒ혜리미 안녕하세요 여행 인플루언서 혜리미입니다 어제... 부모님 모시고 가기 좋은 힐링 포인트 앵지밭골그집은 부모님과 함께 가기 딱 좋아요 이유는 주인장의...
   https://blog.naver.com/hyelimev_v/224240908267

• 창원 마산 대형카페... 주차까지 편해 부모님 모시고 가기 좋은 곳
   #공간언저리 #마산진동카페 #창원대형카페 #마산뷰카페 #창원아이랑가볼만한곳 #마산부모님모시고가기좋은곳 #마산말차라떼맛집 #마산진동맛집 #마산키즈케어존 #창원나들이
   https://blog.naver.com/sprisoh/224286255490

• 창원 해신탕 먹물집 본점, 부모님 모시고 가기 좋은 몸보신 코스
   부모님 모시고 방문하기에도 부담이 없었어요. 먹물집은 로데오 주차빌딩 2층에 있어요. 저희는 해신탕을 먹기로했지만, 동해안 활문어 샤브코스도 좋아보여요. 창원 몸보신에 좋은 해신탕 먹물집에 가니 넓은...
   https://blog

In [ ]:
import nest_asyncio, os, httpx, html, re
from pydantic import BaseModel, Field
from google.adk.agents import LlmAgent
from google.adk.tools.agent_tool import AgentTool
from google.adk.runners import InMemoryRunner
from google.genai import types
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()
MODEL = "gemini-3.6-flash"
CID = os.environ["NAVER_CLIENT_ID"]
CSEC = os.environ["NAVER_CLIENT_SECRET"]

# ══════════════════════════════════════════════════
# JSON 정리 에이전트 — tools 없음, output_schema만
# ══════════════════════════════════════════════════
class Card(BaseModel):
    domain: str = Field(description="restaurant/cafe/festival/spot 중 하나")
    name: str = Field(description="장소/행사 이름")
    reason: str = Field(description="추천 이유 한 줄")
    source: str = Field(description="근거 URL")

class Feed(BaseModel):
    cards: list[Card] = Field(description="추천 카드 6개")

formatter = LlmAgent(
    name="formatter", model=MODEL,
    instruction=(
        "주어진 추천 텍스트를 카드 JSON으로 변환해라.\n"
        "- 근거 URL이 없는 항목은 제외.\n"
        "- 최대 6개.\n"
        "- domain은 restaurant/cafe/festival/spot 중 하나로 분류."
    ),
    output_schema=Feed,
)

# ── formatter 실행 헬퍼 ──
async def to_json(text: str) -> Feed:
    runner = InMemoryRunner(agent=formatter, app_name="wh")
    s = await runner.session_service.create_session(app_name="wh", user_id="u1")
    out = ""
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        if ev.is_final_response() and ev.content:
            out = "".join(x.text or "" for x in ev.content.parts)
    return Feed.model_validate_json(out)

# ══════════════════════════════════════════════════
# 검색 함수툴 — 서브 에이전트가 이걸 부른다
# ══════════════════════════════════════════════════
def _strip(t):
    return html.unescape(re.sub(r"<[^>]+>", "", t or "")).strip()

def naver_search(query: str, kind: str = "blog") -> list:
    """네이버에서 정보를 검색한다.

    Args:
        query: 검색어. 지역명을 포함하면 좋다. 예 '창원 축제 2026'
        kind: 'blog'(후기), 'local'(장소·주소), 'news'(공지) 중 하나
    """
    r = httpx.get(
        f"https://openapi.naver.com/v1/search/{kind}.json",
        headers={"X-Naver-Client-Id": CID, "X-Naver-Client-Secret": CSEC},
        params={"query": query, "display": 5, "sort": "sim"},
        timeout=5.0,
    )
    r.raise_for_status()
    return [
        {"title": _strip(it.get("title","")),
         "desc": _strip(it.get("description",""))[:150],
         "url": it.get("link","")}
        for it in r.json().get("items", [])
    ]
    
def _naver_real_url(url: str) -> str:
    """네이버 블로그는 iframe이라 그대로 긁으면 본문이 없다.
    blog.naver.com/{id}/{logNo} → PostView 로 바꿔야 진짜 HTML이 나온다."""
    m = re.search(r"blog\.naver\.com/([^/]+)/(\d+)", url)
    if m:
        return f"https://blog.naver.com/PostView.naver?blogId={m.group(1)}&logNo={m.group(2)}"
    return url

def fetch_page(url: str) -> str:
    """웹 페이지를 열어 본문 텍스트를 가져온다.

    naver_search로 찾은 URL 중 더 자세히 볼 것을 이걸로 읽는다.
    검색 스니펫만으로 부족할 때 사용해라.

    Args:
        url: 읽을 페이지의 전체 URL
    """
    UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
    try:
        r = httpx.get(_naver_real_url(url), headers={"User-Agent": UA},
                      timeout=10.0, follow_redirects=True)
        r.raise_for_status()
    except httpx.HTTPError as e:
        return f"[열기 실패: {type(e).__name__}] 이 URL은 건너뛰세요."

    raw = r.text
    # script/style 제거 후 태그 제거
    body = re.sub(r"<(script|style|head)[^>]*>.*?</\1>", " ", raw, flags=re.S|re.I)
    text = _strip(re.sub(r"<[^>]+>", " ", body))
    return text[:1500] if len(text) > 200 else f"[본문 추출 실패, {len(text)}자만 나옴]"

# ══════════════════════════════════════════════════
# 서브 에이전트 4개 — 각자 naver_search 툴 보유
# ══════════════════════════════════════════════════
def make_searcher(name, desc, hint):
    return LlmAgent(
        name=name, model=MODEL,
        description=f"{desc}을(를) 네이버에서 검색한다.",
        instruction=(
            f"너는 {desc} 검색 전문이다.\n"
            f"1. naver_search 툴로 검색해라. ({hint})\n"
            "2. 검색 결과 중 사용자 조건에 맞아 보이는 2~3개는 "
            "fetch_page로 본문을 읽어 자세히 확인해라.\n"
            "3. 본문에서 확인한 내용으로 [이름 / 특징 / 근거URL] 정리해라.\n"
            "검색 결과나 본문에 없는 건 지어내지 마라."
        ),
        tools=[naver_search, fetch_page],   # ← 툴 2개
    )

restaurant = make_searcher("restaurant_agent", "식당·맛집", "kind='blog'나 'local' 사용")
cafe       = make_searcher("cafe_agent", "카페·디저트", "kind='blog'나 'local' 사용")
festival   = make_searcher("festival_agent", "축제·행사·공연", "kind='blog'나 'news' 사용")
spot       = make_searcher("spot_agent", "관광지·명소", "kind='blog'나 'local' 사용")

# ══════════════════════════════════════════════════
# 메인 에이전트 — 서브를 AgentTool로 호출 (위임)
# ══════════════════════════════════════════════════
main_agent = LlmAgent(
    name="main", model=MODEL,
    instruction=(
        "너는 지역 추천 코디네이터다.\n"
        "1. 사용자 요청을 보고 관련된 검색 에이전트를 호출해라.\n"
        "   (맛집→restaurant_agent, 카페→cafe_agent, 축제/행사→festival_agent, 관광지→spot_agent)\n"
        "2. 요청 주제에 맞는 에이전트를 하나 이상 호출해라.\n"
        "3. 결과를 모아 조건에 맞는 것 6개를 골라 정리해라.\n"
        "4. 각 항목을 [도메인 | 이름 | 이유 | URL] 로 보여줘라. 근거 URL 없는 건 빼라."
    ),
    tools=[
        AgentTool(agent=restaurant),
        AgentTool(agent=cafe),
        AgentTool(agent=festival),
        AgentTool(agent=spot),
    ],
)

# ══════════════════════════════════════════════════
# 실행 + 로그
# ══════════════════════════════════════════════════


async def run_main(user_request: str):
    runner = InMemoryRunner(agent=main_agent, app_name="wh")
    s = await runner.session_service.create_session(app_name="wh", user_id="u1")
    print(f"[USER] {user_request}\n{'='*60}")
    final = ""
    async for ev in runner.run_async(
        user_id="u1", session_id=s.id,
        new_message=types.Content(role="user", parts=[types.Part(text=user_request)]),
    ):
        for p in (ev.content.parts if ev.content else []) or []:
            if getattr(p, "function_call", None):
                fc = p.function_call
                print(f"  ↪ [{ev.author}] 호출: {fc.name}({dict(fc.args)})")
            elif getattr(p, "function_response", None):
                print(f"  ← [{ev.author}] {p.function_response.name} 응답")
        if ev.is_final_response() and ev.content:
            final = "".join(x.text or "" for x in ev.content.parts)
    print(f"\n{'='*60}\n[최종]\n{final}")
    return final

async def run_full(user_request: str):
    text = await run_main(user_request)          # 1~5단계 (위임+검색+정리 텍스트)

    print(f"\n{'='*60}\n[6단계] JSON 변환 중...")
    feed = await to_json(text)                    # 6단계: 텍스트 → JSON

    print(f"\n[카드 {len(feed.cards)}개]")
    for c in feed.cards:
        print(f"  [{c.domain}] {c.name}")
        print(f"       {c.reason}")
        print(f"       {c.source}")
    return feed

feed = await run_full("부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이.")

# 진짜 JSON으로 쓸 수 있는지 확인
print("\n=== JSON 객체 접근 ===")
print(feed.cards[0].name)
print(feed.model_dump_json(indent=2)[:400])

[USER] 부모님 모시고 가기 좋은 축제 찾아줘. 창원이나 창원 근교, 25일에서 27일 사이.
  ↪ [main] 호출: festival_agent({'request': '창원 축제 10월 25일 26일 27일'})
  ↪ [main] 호출: festival_agent({'request': '마산국화축제'})
  ← [main] festival_agent 응답
  ← [main] festival_agent 응답
  ↪ [main] 호출: festival_agent({'request': '김해 축제 10월 25일 26일 27일'})
  ↪ [main] 호출: festival_agent({'request': '함안 축제 10월 25일 26일 27일'})
  ↪ [main] 호출: festival_agent({'request': '진주 축제 10월 25일 26일 27일'})
  ← [main] festival_agent 응답
  ← [main] festival_agent 응답
  ← [main] festival_agent 응답

[최종]
부모님을 모시고 방문하기 좋은 창원 및 창원 근교(함안, 진주)의 10월 25일~27일 기간 주요 축제 6곳을 추천해 드립니다.

* [축제·행사 | 마산가고파국화축제 | 수십만 송이의 다채로운 국화 조형물과 해양 공원의 바다 풍경이 어우러져 부모님과 화사하게 꽃구경하며 산책하기 최적입니다. | https://blog.naver.com/sl7856/224062275737]
* [축제·행사 | 창원 단감축제 | 창원 특산물인 맛있는 단감을 맛보고 다채로운 시식 행사와 공연을 즐길 수 있는 정겨운 향토 축제입니다. | https://blog.naver.com/rlarlwjdsla12/223633824681]
* [축제·행사 | 창원조각비엔날레 | 창원시 주요 전시 공간 및 야외 공원에서 조각 작품을 감상할 수 있어 부모님과 여유롭고 품격 있게 산책하기 좋습니다. | https://www.meconomynews.com/news/art